**Last code edit:** 2026-08-16 14:56 (UTC+03:00)

In [14]:
import os
import shutil
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

folder_name = 'biblioteca'

# 1. Reset directory
if os.path.exists(folder_name):
    shutil.rmtree(folder_name)

os.makedirs(folder_name)

# 2. Load dataset
ut_df = pd.read_csv('clean/user_taggedartists_clean.csv')

# 3. Restrict the corpus to content-eligible tags
# Module 0 (Chapter 23) marks a tag ineligible when it sits on fewer than two
# distinct artists, or when it describes preference or opinion rather than
# music. Set to False to rebuild the unfiltered corpus for comparison.
FILTER_TO_CONTENT_MODEL_TAGS = True

if FILTER_TO_CONTENT_MODEL_TAGS:
    corpus_df = ut_df[ut_df['use_for_content_model']]
else:
    corpus_df = ut_df

# 4. Group by artistID and write tags to individual files
file_extension = '.txt'

for artist_id, group in corpus_df.groupby('artistID'):
    filepath = os.path.join(folder_name, f"{artist_id}{file_extension}")
    # Extract canonical tags and write them (one per line)
    tags = group['canonical_tag'].dropna().tolist()
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(tags) + '\n')

print(f"Successfully populated files for {corpus_df['artistID'].nunique()} artists in '{folder_name}'.")

print(f"\n--- Corpus filtering (FILTER_TO_CONTENT_MODEL_TAGS={FILTER_TO_CONTENT_MODEL_TAGS}) ---")
print(f"Vocabulary: {ut_df['canonical_tag'].nunique()} tags -> {corpus_df['canonical_tag'].nunique()}")
print(f"Tag votes:  {len(ut_df)} -> {len(corpus_df)}")
print(f"Artists:    {ut_df['artistID'].nunique()} -> {corpus_df['artistID'].nunique()} "
      f"({ut_df['artistID'].nunique() - corpus_df['artistID'].nunique()} left with no eligible tag)")

# ---
# 4.1
# ---

distinct_values_list = sorted(
    corpus_df.iloc[1:, 3].dropna().unique().tolist()
)

with open('tag_list.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(distinct_values_list) + '\n')


# ---------------------------------------------------------
# 5. Build TF-IDF Sparse Matrix
# ---------------------------------------------------------

# Collect file paths and retain order of artist IDs
file_paths = []
artist_ids = []

for filename in sorted(os.listdir(folder_name)):
    if filename.endswith(file_extension):
        file_paths.append(os.path.join(folder_name, filename))
        artist_ids.append(filename.replace(file_extension, ''))


def newline_tokenizer(text):
    return [line.strip() for line in text.splitlines() if line.strip()]

# Initialize TfidfVectorizer: split strictly by newline characters
vectorizer = TfidfVectorizer(
    input='filename',
    tokenizer=newline_tokenizer,
    lowercase=True,
    token_pattern=None
)

# Fit and transform to create the sparse matrix
tfidf_matrix = vectorizer.fit_transform(file_paths)

# Inspect the result
print("\n--- TF-IDF Matrix Built ---")
print(f"Matrix shape (Artists x Unique Tags): {tfidf_matrix.shape}")
print(f"Matrix type: {type(tfidf_matrix)}")  # scipy.sparse._csr.csr_matrix



# Get feature names (tags)
feature_names = vectorizer.get_feature_names_out()

print(f"Total Unique Tags: {len(feature_names)}")
print("First 10 tags:", feature_names[:10])


target_artist_id = "52"  # Change to any artistID in your dataset

if target_artist_id in artist_ids:
    # 1. Find row index in the list
    row_idx = artist_ids.index(target_artist_id)
    
    # 2. Extract sparse row using .getrow() to avoid direct bracket indexing errors
    artist_row = tfidf_matrix.getrow(row_idx)
    
    # 3. Get non-zero column indices and their corresponding TF-IDF values
    _, col_indices = artist_row.nonzero()
    scores = artist_row.data
    
    # 4. Map back to tag names
    feature_names = vectorizer.get_feature_names_out()
    
    df_artist = pd.DataFrame({
        'tag': feature_names[col_indices],
        'tfidf': scores
    }).sort_values(by='tfidf', ascending=False)

    print(f"\n--- Top Tags for Artist ID: {target_artist_id} ---")
    print(df_artist.head(10).to_string(index=False))
else:
    print(f"Artist ID {target_artist_id} not found.")


import textwrap

# Join all names into a single string separated by spaces (or commas)
full_text = " ".join(feature_names)

# Wrap to a maximum of 140 characters per line
wrapped_lines = textwrap.wrap(full_text, width=140)

for line in wrapped_lines:
    print(line)

# 147 Total Unique Tags: 9482
# 154                    9476
# 155                    9475
# 161                    9470
# 162                    9469
# 163                    9468

def cleanup_saved_files():
    """Deletes the TF-IDF matrix, vectorizer, and artist IDs files if they exist."""
    files_to_delete = [
        'tfidf_matrix.npz',
        'vectorizer.joblib',
        'artist_ids.json'
    ]
    for file_path in files_to_delete:
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Deleted: {file_path}")
        else:
            print(f"Not found (skipped): {file_path}")

cleanup_saved_files()

import json
import joblib
from scipy.sparse import save_npz
# 1. Save SciPy sparse matrix (.npz is faster & more compact than pickle)
save_npz('tfidf_matrix.npz', tfidf_matrix)
# 2. Save the fitted TfidfVectorizer
joblib.dump(vectorizer, 'vectorizer.joblib')
# 3. Save artist_ids list to maintain exact row ordering
with open('artist_ids.json', 'w', encoding='utf-8') as f:
    json.dump(artist_ids, f)
print("Static artifacts successfully saved to disk!")

Successfully populated files for 11839 artists in 'biblioteca'.

--- Corpus filtering (FILTER_TO_CONTENT_MODEL_TAGS=True) ---
Vocabulary: 9297 tags -> 3500
Tag votes:  182133 -> 163418
Artists:    12130 -> 11839 (291 left with no eligible tag)

--- TF-IDF Matrix Built ---
Matrix shape (Artists x Unique Tags): (11839, 3500)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Total Unique Tags: 3500
First 10 tags: ['-pearl fashion music' '00' '1' '1008' '10s' '1337' '1337 guitar players'
 '1940s' '1950s' '1960s']

--- Top Tags for Artist ID: 52 ---
            tag    tfidf
       trip hop 0.760121
      chill out 0.411367
      downtempo 0.341081
     electronic 0.183637
female vocalist 0.134398
         lounge 0.113514
        the end 0.089058
        science 0.085926
     drug music 0.084248
           btvs 0.083497
-pearl fashion music 00 1 1008 10s 1337 1337 guitar players 1940s 1950s 1960s 1963 1964 1965 1966 1967 1968 1969 1970 1970s 1971 1973 1978
1979 1980 1980s 1981 1982 1983 19

In [17]:
# Retrieval examples: rank artists by cosine similarity to tag queries.
# Multi-tag searches use one line per tag because the vectorizer's tokenizer
# treats each line as a complete tag.
import numpy as np
from IPython.display import display

artists_df = pd.read_csv('clean/artists_clean.csv')
artist_name_by_id = (
    artists_df.assign(artistID=artists_df['artistID'].astype(str))
    .set_index('artistID')['source_name_repaired']
)


def retrieve_artists(query_tags, top_k=10):
    """Return the artists whose TF-IDF tag profiles best match query_tags."""
    query_tags = [tag.strip().lower() for tag in query_tags if tag.strip()]
    known_tags = set(vectorizer.get_feature_names_out())
    unknown_tags = [tag for tag in query_tags if tag not in known_tags]
    if unknown_tags:
        raise ValueError(f"Tags not present in the TF-IDF vocabulary: {unknown_tags}")

    vectorizer.input = 'content'
    query_vector = vectorizer.transform(['\n'.join(query_tags)])
    # TfidfVectorizer L2-normalizes vectors, so this dot product is cosine similarity.
    scores = (tfidf_matrix @ query_vector.T).toarray().ravel()
    ranked_indices = np.argsort(scores)[::-1]
    ranked_indices = [idx for idx in ranked_indices if scores[idx] > 0][:top_k]

    return pd.DataFrame({
        'rank': range(1, len(ranked_indices) + 1),
        'artistID': [artist_ids[idx] for idx in ranked_indices],
        'artist': [artist_name_by_id.get(artist_ids[idx], '<unknown>') for idx in ranked_indices],
        'cosine_similarity': [scores[idx] for idx in ranked_indices],
    })


retrieval_examples = {
    'russian rock': ['russian rock'],
    'funk': ['funk'],
    # this one is the most closes to the task's 
    # melancholic indie folk
    'upbeat indie rock': ['upbeat', 'indie rock'],
}

for query, tags in retrieval_examples.items():
    print(f"\nTop artists for {query!r} (tags: {', '.join(tags)})")
    display(retrieve_artists(tags, top_k=10))


Top artists for 'russian rock' (tags: russian rock)


,rank,artistID,artist,cosine_similarity
0,1,18384,Захар Май,1.000000
1,2,18385,Разные Люди,1.000000
2,3,4906,Сурганова и Оркестр,1.000000
3,4,4905,Мультfильмы,0.932189
4,5,4299,Браво,0.932189
5,6,1796,Nautilus Pompilius,0.932189
6,7,7008,Пилот,0.860984
7,8,3837,Алиса,0.828616
8,9,3046,Zемфира,0.784908
9,10,11170,Король и Шут,0.710198



Top artists for 'funk' (tags: funk)


,rank,artistID,artist,cosine_similarity
0,1,8226,Skeewiff,1.000000
1,2,18259,MC Gi,1.000000
2,3,1823,MFSB,1.000000
3,4,7574,The Daktaris,1.000000
4,5,10377,Alain Goraguer,1.000000
5,6,14217,Cymande,0.959639
6,7,7584,Oneness of Juju,0.917862
7,8,1230,Parliament,0.897981
8,9,13926,The Bongolian,0.885165
9,10,13815,The Bar-Kays,0.862728



Top artists for 'upbeat indie rock' (tags: upbeat, indie rock)


,rank,artistID,artist,cosine_similarity
0,1,5316,Iglu & Hartly,0.497209
1,2,15451,Ultrabeat,0.476528
2,3,12410,Howling Bells,0.441540
3,4,18459,British India,0.441540
4,5,3770,Julian Plenti,0.441540
5,6,13419,Trembling Blue Stars,0.441540
6,7,12487,Empire! Empire! (I Was A Lonely Estate),0.441540
7,8,7889,Mint,0.441540
8,9,15344,JJ72,0.441540
9,10,10039,A Silent Film,0.441540
